# Snapshot-based 2D--3D flow agreement with combined comparison panels

- submesoscale-active area fraction \(A_{\mathrm{sub}} = \Pr(|\zeta/f|>0.5)\),
- normalized-vorticity distribution,
- normalized-strain distribution,
- 3D-only divergence proxy,
- Wasserstein distances for vorticity and strain.


In [ ]:
# ============================================================
# 1. IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path
import sys
import json
import importlib
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from scipy.stats import wasserstein_distance


NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent
FLOW_DIR = PROJECT_DIR / "z.flow_postprocessing"
FLOW_SCRIPTS_DIR = FLOW_DIR / "scripts"

for path in [PROJECT_DIR, FLOW_SCRIPTS_DIR]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

import theme.plot_theme as ptheme
import shcherbina_utils as shu

importlib.reload(ptheme)
ptheme.apply_theme()

importlib.reload(shu)
shu.apply_plot_style()

print(f"Notebook dir     : {NB_DIR}")
print(f"Project dir      : {PROJECT_DIR}")
print(f"Flow scripts dir : {FLOW_SCRIPTS_DIR}")
print(f"Utilities        : {Path(shu.__file__).resolve()}")


In [ ]:
# ============================================================
# 2. CASE AND SNAPSHOT SETTINGS
# ============================================================

CASE_3D = "run_jun1"
CASE_2D = f"{CASE_3D}_2D"

INPUT_DIR = (NB_DIR / "../data/input").resolve()
RESULTS_ROOT = (NB_DIR / "../results").resolve()

DATA_FILE_3D = INPUT_DIR / f"{CASE_3D}.nc"
DATA_FILE_2D = INPUT_DIR / f"{CASE_2D}.nc"

SNAPSHOT_DAYS = [0.0, 2.0, 4.0, 6.0, 8.0]
SNAPSHOT_TOLERANCE_DAYS = None

DAY_PER_INDEX_3D = 0.5
DAY_PER_INDEX_2D = 0.5

REFERENCE_START_INDEX_3D = None
REFERENCE_START_INDEX_2D = None
DEFAULT_REFERENCE_START_INDEX = 0

LEVEL_INDICES_3D = (0,)
LEVEL_INDICES_2D = (0,)

DX_3D = None
DY_3D = None
DX_2D = None
DY_2D = None

F0 = 8.0e-5
ACTIVE_RO_THRESHOLD = 0.5
MAX_DISTRIBUTION_SAMPLES = 150_000

NBINS_PDF = 80
NBINS_JPDF = 100
PDF_PERCENTILE_LIMITS = (0.1, 99.9)

SUMMARY_CASES_3D = None
EXCLUDE_SUMMARY_CASES_3D = []

SAVE_OUTPUTS = True
SAVE_FIGURES = True

MAKE_SELECTED_CASE_MAPS = True
MAKE_SELECTED_CASE_PDFS = True
MAKE_SELECTED_CASE_JPDFS = True

SNAPSHOT_TAG = "-".join(
    f"{float(day):g}".replace(".", "p")
    for day in SNAPSHOT_DAYS
)

COMPARISON_ID = (
    f"{CASE_3D}_vs_{CASE_2D}_snapshots_combined_{SNAPSHOT_TAG}d"
)

OUTPUT_DIR = RESULTS_ROOT / "comparisons" / COMPARISON_ID
METRICS_DIR = OUTPUT_DIR / "metrics"
FIG_DIR = OUTPUT_DIR / "figures"
MAP_DIR = FIG_DIR / "map_panels"
PDF_DIR = FIG_DIR / "pdf_panels"
JPDF_DIR = FIG_DIR / "jpdf_panels"

if SAVE_OUTPUTS:
    METRICS_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    MAP_DIR.mkdir(parents=True, exist_ok=True)
    PDF_DIR.mkdir(parents=True, exist_ok=True)
    JPDF_DIR.mkdir(parents=True, exist_ok=True)

print(f"3D input       : {DATA_FILE_3D}")
print(f"2D input       : {DATA_FILE_2D}")
print(f"Snapshot days  : {SNAPSHOT_DAYS}")
print(f"Output dir     : {OUTPUT_DIR}")


In [ ]:
# ============================================================
# 3. HELPERS
# ============================================================

def newest_collection_config(case_name):
    metadata_dir = RESULTS_ROOT / case_name / "metadata"
    candidates = sorted(
        metadata_dir.glob("*_config.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        return None
    return candidates[-1]


def optional_case_config(case_name):
    config_path = newest_collection_config(case_name)

    if config_path is None:
        return {}

    try:
        with config_path.open("r", encoding="utf-8") as file:
            config = json.load(file)
    except (OSError, json.JSONDecodeError) as error:
        warnings.warn(
            f"Could not read {config_path}: {error}. "
            "The fallback reference index will be used."
        )
        return {}

    config["_config_path"] = str(config_path)
    return config


def resolve_reference_index(config, manual_index):
    if manual_index is not None:
        return int(manual_index)
    return int(config.get("release_time_index", DEFAULT_REFERENCE_START_INDEX))


def elapsed_days(n_times, day_per_index, reference_start_index):
    return (
        np.arange(int(n_times), dtype=float) - int(reference_start_index)
    ) * float(day_per_index)


def nearest_snapshot_indices(time_days, requested_days, tolerance_days=None):
    time_days = np.asarray(time_days, dtype=float)
    requested_days = np.asarray(requested_days, dtype=float)

    if tolerance_days is None:
        finite_times = time_days[np.isfinite(time_days)]
        if finite_times.size > 1:
            output_interval = float(np.nanmedian(np.diff(finite_times)))
            tolerance_days = 0.51 * abs(output_interval)
        else:
            tolerance_days = 1.0e-8

    selections = []
    used_indices = set()

    for requested_day in requested_days:
        index = int(np.nanargmin(np.abs(time_days - requested_day)))
        actual_day = float(time_days[index])
        difference = abs(actual_day - float(requested_day))

        if difference > float(tolerance_days):
            raise ValueError(
                f"Requested snapshot T={requested_day:g} d has no output within "
                f"{tolerance_days:g} d. Nearest output is T={actual_day:g} d."
            )

        if index in used_indices:
            raise ValueError(
                f"Two requested times map to the same output index {index}."
            )

        used_indices.add(index)
        selections.append(
            {
                "requested_day": float(requested_day),
                "actual_day": actual_day,
                "time_index": index,
            }
        )

    return selections


def discover_2d_3d_case_pairs(input_dir):
    input_dir = Path(input_dir)
    cases = []

    for path_3d in sorted(input_dir.glob("run_*.nc")):
        case_3d = path_3d.stem
        if case_3d.endswith("_2D"):
            continue

        path_2d = input_dir / f"{case_3d}_2D.nc"
        if path_2d.exists():
            cases.append(case_3d)

    return cases


def infer_grid_spacing(ds, coordinate_name, fallback=None):
    if coordinate_name in ds.coords:
        coordinate = np.asarray(ds[coordinate_name].values, dtype=float)
        if coordinate.size > 1:
            spacing = float(np.nanmedian(np.abs(np.diff(coordinate))))
            if np.isfinite(spacing) and spacing > 0.0:
                return spacing

    if fallback is not None:
        return float(fallback)

    raise ValueError(f"Could not infer grid spacing from {coordinate_name}.")


def dataset_layout(ds):
    time_dim = shu.find_dim(ds["UVEL"].dims, ["T", "time", "iter"])
    z_dim = shu.find_dim(ds["UVEL"].dims, ["Z", "Zmd", "k", "depth"])
    return time_dim, z_dim


def calculate_kinematics(ds, time_dim, z_dim, time_index, level_indices, dx, dy):
    ro, div_f, strain_f = shu.compute_surface_kinematics(
        ds,
        time_dim=time_dim,
        z_dim=z_dim,
        it=int(time_index),
        levels=list(level_indices),
        dx=float(dx),
        dy=float(dy),
        f0=float(F0),
    )
    return {
        "ro": np.asarray(ro, dtype=float),
        "div_f": np.asarray(div_f, dtype=float),
        "strain_f": np.asarray(strain_f, dtype=float),
    }


def full_domain_active_fraction(ro, threshold=ACTIVE_RO_THRESHOLD):
    ro = np.asarray(ro, dtype=float)
    ny, nx = ro.shape

    full_domain = [
        {
            "square_id": 1,
            "x0": 0,
            "x1": nx,
            "y0": 0,
            "y1": ny,
        }
    ]

    result = shu.compute_pro_per_square(ro, full_domain, threshold)
    return float(result[0]["PRo"])


def deterministic_subsample(values, max_samples):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if max_samples is None or values.size <= int(max_samples):
        return values

    step = int(np.ceil(values.size / int(max_samples)))
    return values[::step]


def distribution_distance(values_2d, values_3d, max_samples=MAX_DISTRIBUTION_SAMPLES):
    values_2d = deterministic_subsample(values_2d, max_samples)
    values_3d = deterministic_subsample(values_3d, max_samples)

    if values_2d.size == 0 or values_3d.size == 0:
        return np.nan, np.nan

    raw = float(wasserstein_distance(values_2d, values_3d))

    q25, q75 = np.nanpercentile(values_3d, [25.0, 75.0])
    iqr_3d = float(q75 - q25)

    normalized = raw / iqr_3d if np.isfinite(iqr_3d) and iqr_3d > 0.0 else np.nan
    return raw, normalized


def divergence_rms_3d(div_f):
    div_f = np.asarray(div_f, dtype=float)
    finite = np.isfinite(div_f)
    if not np.any(finite):
        return np.nan
    return float(np.sqrt(np.nanmean(div_f[finite] ** 2)))


def common_pdf_edges(arrays, n_bins=NBINS_PDF, percentile_limits=PDF_PERCENTILE_LIMITS, lower_bound=None):
    finite_arrays = []

    for values in arrays:
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size:
            finite_arrays.append(values)

    if not finite_arrays:
        raise ValueError("No finite values are available for PDF binning.")

    combined = np.concatenate(finite_arrays)
    lower, upper = np.nanpercentile(combined, percentile_limits)

    if lower_bound is not None:
        lower = max(float(lower_bound), float(lower))

    if upper <= lower:
        upper = lower + 1.0

    padding = 0.03 * (upper - lower)

    if lower_bound is None:
        lower_edge = lower - padding
    else:
        lower_edge = max(float(lower_bound), lower - padding)

    upper_edge = upper + padding
    return np.linspace(lower_edge, upper_edge, int(n_bins) + 1)


def common_jpdf_edges(ro_3d, strain_3d, ro_2d, strain_2d, n_bins=NBINS_JPDF):
    ro_all = np.concatenate(
        [
            np.asarray(ro_3d, dtype=float).ravel(),
            np.asarray(ro_2d, dtype=float).ravel(),
        ]
    )
    strain_all = np.concatenate(
        [
            np.asarray(strain_3d, dtype=float).ravel(),
            np.asarray(strain_2d, dtype=float).ravel(),
        ]
    )

    ro_all = ro_all[np.isfinite(ro_all)]
    strain_all = strain_all[np.isfinite(strain_all)]

    xlow, xhigh = np.nanpercentile(ro_all, [0.2, 99.8])
    ylow, yhigh = np.nanpercentile(strain_all, [0.0, 99.8])

    xpad = 0.05 * (xhigh - xlow)
    ypad = 0.05 * (yhigh - ylow)

    x_edges = np.linspace(xlow - xpad, xhigh + xpad, n_bins + 1)
    y_edges = np.linspace(max(0.0, ylow - ypad), yhigh + ypad, n_bins + 1)
    return x_edges, y_edges


def normalized_histogram(values, edges):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    hist, _ = np.histogram(values, bins=edges, density=True)
    return shu.normalize_hist(hist), hist


def regime_fractions(ro, strain):
    ro = np.asarray(ro, dtype=float)
    strain = np.asarray(strain, dtype=float)

    mask = np.isfinite(ro) & np.isfinite(strain)
    if not np.any(mask):
        return {"AVD": np.nan, "SD": np.nan, "CVD": np.nan}

    ro = ro[mask]
    strain = strain[mask]

    avd = (ro < 0.0) & (np.abs(ro) > strain)
    cvd = (ro > 0.0) & (np.abs(ro) > strain)
    sd = ~(avd | cvd)

    total = float(ro.size)
    return {
        "AVD": np.sum(avd) / total,
        "SD": np.sum(sd) / total,
        "CVD": np.sum(cvd) / total,
    }


def safe_filename_day(day):
    return f"{float(day):g}".replace("-", "m").replace(".", "p")


In [ ]:
# ============================================================
# 4. COMBINED PANEL PLOTTING HELPERS
# ============================================================

def plot_side_by_side_field_panel(
    field_3d,
    field_2d,
    dx,
    dy,
    title_left,
    title_right,
    suptitle,
    cbar_label,
    cmap,
    vmin,
    vmax,
    output_path=None,
):
    shu.apply_plot_style()

    if isinstance(cmap, str):
        cmap = shu.get_cmap(cmap)

    field_3d = np.asarray(field_3d, dtype=float)
    field_2d = np.asarray(field_2d, dtype=float)

    ny = min(field_3d.shape[0], field_2d.shape[0])
    nx = min(field_3d.shape[1], field_2d.shape[1])

    field_3d = field_3d[:ny, :nx]
    field_2d = field_2d[:ny, :nx]

    x_edges = np.arange(nx + 1) * dx / 1000.0
    y_edges = np.arange(ny + 1) * dy / 1000.0

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5.5),
        sharex=True,
        sharey=True,
        facecolor="white",
    )

    for ax, field, title in zip(
        axes,
        [field_3d, field_2d],
        [title_left, title_right],
    ):
        ax.set_facecolor("white")
        pcm = ax.pcolormesh(
            x_edges,
            y_edges,
            field,
            shading="flat",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        shu.format_axis(
            ax,
            title=title,
            xlabel="x [km]",
            ylabel="y [km]",
            grid=True,
            equal=True,
        )

    cbar = fig.colorbar(
        pcm,
        ax=axes.ravel().tolist(),
        pad=0.03,
        fraction=0.045,
    )
    cbar.set_label(cbar_label)

    fig.suptitle(suptitle, y=1.02)
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(
            output_path,
            dpi=getattr(ptheme, "SAVE_DPI", 300),
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )

    plt.show()


def plot_combined_pdf_panel(
    ro_3d,
    ro_2d,
    strain_3d,
    strain_2d,
    metrics_row,
    title,
    output_path=None,
):
    shu.apply_plot_style()

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 8),
        facecolor="white",
    )

    ro_edges = common_pdf_edges(
        [ro_3d, ro_2d],
        n_bins=NBINS_PDF,
        percentile_limits=PDF_PERCENTILE_LIMITS,
        lower_bound=None,
    )
    strain_edges = common_pdf_edges(
        [strain_3d, strain_2d],
        n_bins=NBINS_PDF,
        percentile_limits=PDF_PERCENTILE_LIMITS,
        lower_bound=0.0,
    )

    shu.plot_pdf_panel(
        axes[0, 0],
        np.ravel(ro_3d),
        bins=ro_edges,
        title="3D vorticity",
        xlabel=r"$\zeta/f$",
        color_line=shu.get_color("highlight"),
        show_zero_line=True,
    )
    shu.plot_pdf_panel(
        axes[0, 1],
        np.ravel(ro_2d),
        bins=ro_edges,
        title="2D vorticity",
        xlabel=r"$\zeta/f$",
        color_line=shu.get_color("secondary"),
        show_zero_line=True,
    )
    shu.plot_pdf_panel(
        axes[1, 0],
        np.ravel(strain_3d),
        bins=strain_edges,
        title="3D strain",
        xlabel=r"$s/f$",
        color_line=shu.get_color("highlight"),
        show_zero_line=False,
    )
    shu.plot_pdf_panel(
        axes[1, 1],
        np.ravel(strain_2d),
        bins=strain_edges,
        title="2D strain",
        xlabel=r"$s/f$",
        color_line=shu.get_color("secondary"),
        show_zero_line=False,
    )

    s_ro_3d = shu.stats_dict(np.ravel(ro_3d))
    s_ro_2d = shu.stats_dict(np.ravel(ro_2d))
    s_st_3d = shu.stats_dict(np.ravel(strain_3d))
    s_st_2d = shu.stats_dict(np.ravel(strain_2d))

    textbox = (
        rf"$A_{{sub}}^{{3D}}={metrics_row['active_fraction_3d']:.3f}$   "
        rf"$A_{{sub}}^{{2D}}={metrics_row['active_fraction_2d']:.3f}$   "
        rf"$\Delta A_{{sub}}={metrics_row['active_fraction_error_signed']:.3f}$"
        "\n"
        rf"Vorticity: $W_1={metrics_row['vorticity_wasserstein']:.3f}$, "
        rf"$W_1^*={metrics_row['vorticity_wasserstein_iqr_normalized']:.3f}$, "
        rf"$\Delta \mu={s_ro_2d['mean'] - s_ro_3d['mean']:.3f}$, "
        rf"$\Delta \sigma={s_ro_2d['std'] - s_ro_3d['std']:.3f}$"
        "\n"
        rf"Strain: $W_1={metrics_row['strain_wasserstein']:.3f}$, "
        rf"$W_1^*={metrics_row['strain_wasserstein_iqr_normalized']:.3f}$, "
        rf"$\Delta \mu={s_st_2d['mean'] - s_st_3d['mean']:.3f}$, "
        rf"$\Delta \sigma={s_st_2d['std'] - s_st_3d['std']:.3f}$"
    )

    fig.suptitle(title, y=0.98)
    fig.text(
        0.5,
        0.015,
        textbox,
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(facecolor="white", edgecolor="0.8", alpha=0.95, pad=4),
    )
    fig.tight_layout(rect=[0, 0.07, 1, 0.95])

    if output_path is not None:
        fig.savefig(
            output_path,
            dpi=getattr(ptheme, "SAVE_DPI", 300),
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )

    plt.show()


def make_log_jpdf(ro, strain, x_edges, y_edges):
    ro = np.asarray(ro, dtype=float).ravel()
    strain = np.asarray(strain, dtype=float).ravel()

    mask = np.isfinite(ro) & np.isfinite(strain)
    ro = ro[mask]
    strain = strain[mask]

    H, _, _ = np.histogram2d(
        ro,
        strain,
        bins=[x_edges, y_edges],
        density=True,
    )

    if np.nanmax(H) > 0.0:
        Hn = H / np.nanmax(H)
    else:
        Hn = H.copy()

    with np.errstate(divide="ignore"):
        Hlog = np.log10(Hn)

    Hlog[~np.isfinite(Hlog)] = np.nan
    return H, Hlog


def plot_combined_jpdf_panel(
    ro_3d,
    strain_3d,
    ro_2d,
    strain_2d,
    title,
    output_path=None,
):
    shu.apply_plot_style()

    x_edges, y_edges = common_jpdf_edges(
        ro_3d,
        strain_3d,
        ro_2d,
        strain_2d,
        n_bins=NBINS_JPDF,
    )

    _, Hlog_3d = make_log_jpdf(ro_3d, strain_3d, x_edges, y_edges)
    _, Hlog_2d = make_log_jpdf(ro_2d, strain_2d, x_edges, y_edges)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
        sharex=True,
        sharey=True,
        facecolor="white",
    )

    cmap = shu.get_cmap("jpdf")
    vmin = -2.0
    vmax = 0.0

    for ax, Hlog, panel_title in zip(
        axes,
        [Hlog_3d, Hlog_2d],
        ["3D", "2D"],
    ):
        ax.set_facecolor("white")
        pcm = ax.pcolormesh(
            x_edges,
            y_edges,
            Hlog.T,
            shading="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )

        xx = np.linspace(x_edges[0], x_edges[-1], 500)
        ax.plot(
            xx,
            np.abs(xx),
            "--",
            color=shu.get_color("reference"),
            lw=1.2,
            alpha=0.8,
        )

        shu.add_regime_labels(ax, x_edges, y_edges)
        shu.format_axis(
            ax,
            title=panel_title,
            xlabel=r"$\zeta/f$",
            ylabel=r"$s/f$",
            grid=True,
        )

    cbar = fig.colorbar(
        pcm,
        ax=axes.ravel().tolist(),
        pad=0.03,
        fraction=0.045,
    )
    cbar.set_label(r"$\log_{10}(P/P_{\max})$")

    frac_3d = regime_fractions(ro_3d, strain_3d)
    frac_2d = regime_fractions(ro_2d, strain_2d)

    regime_text = (
        "Regime fractions\n"
        rf"3D: AVD={frac_3d['AVD']:.2f}, SD={frac_3d['SD']:.2f}, CVD={frac_3d['CVD']:.2f}    "
        rf"2D: AVD={frac_2d['AVD']:.2f}, SD={frac_2d['SD']:.2f}, CVD={frac_2d['CVD']:.2f}"
    )

    fig.suptitle(title, y=0.98)
    fig.text(
        0.5,
        0.02,
        regime_text,
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(facecolor="white", edgecolor="0.8", alpha=0.95, pad=4),
    )
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])

    if output_path is not None:
        fig.savefig(
            output_path,
            dpi=getattr(ptheme, "SAVE_DPI", 300),
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )

    plt.show()


In [ ]:
# ============================================================
# 5. LOAD THE SELECTED CASE AND COMPUTE SNAPSHOT METRICS
# ============================================================

if not DATA_FILE_3D.exists():
    raise FileNotFoundError(DATA_FILE_3D)

if not DATA_FILE_2D.exists():
    raise FileNotFoundError(DATA_FILE_2D)

config_3d = optional_case_config(CASE_3D)
config_2d = optional_case_config(CASE_2D)

reference_index_3d = resolve_reference_index(config_3d, REFERENCE_START_INDEX_3D)
reference_index_2d = resolve_reference_index(config_2d, REFERENCE_START_INDEX_2D)

print("Reference indices")
print(f"  3D: {reference_index_3d}")
print(f"  2D: {reference_index_2d}")

selected_records = []
selected_snapshot_data = {}

with xr.open_dataset(
    DATA_FILE_3D,
    decode_times=False,
    cache=False,
) as ds_3d, xr.open_dataset(
    DATA_FILE_2D,
    decode_times=False,
    cache=False,
) as ds_2d:

    time_dim_3d, z_dim_3d = dataset_layout(ds_3d)
    time_dim_2d, z_dim_2d = dataset_layout(ds_2d)

    dx_3d = infer_grid_spacing(ds_3d, "X", fallback=DX_3D)
    dy_3d = infer_grid_spacing(ds_3d, "Y", fallback=DY_3D)
    dx_2d = infer_grid_spacing(ds_2d, "X", fallback=DX_2D)
    dy_2d = infer_grid_spacing(ds_2d, "Y", fallback=DY_2D)

    time_days_3d = elapsed_days(
        ds_3d.sizes[time_dim_3d],
        DAY_PER_INDEX_3D,
        reference_index_3d,
    )
    time_days_2d = elapsed_days(
        ds_2d.sizes[time_dim_2d],
        DAY_PER_INDEX_2D,
        reference_index_2d,
    )

    selections_3d = nearest_snapshot_indices(
        time_days_3d,
        SNAPSHOT_DAYS,
        tolerance_days=SNAPSHOT_TOLERANCE_DAYS,
    )
    selections_2d = nearest_snapshot_indices(
        time_days_2d,
        SNAPSHOT_DAYS,
        tolerance_days=SNAPSHOT_TOLERANCE_DAYS,
    )

    for selection_3d, selection_2d in zip(selections_3d, selections_2d):
        requested_day = selection_3d["requested_day"]

        diagnostics_3d = calculate_kinematics(
            ds_3d,
            time_dim_3d,
            z_dim_3d,
            selection_3d["time_index"],
            LEVEL_INDICES_3D,
            dx_3d,
            dy_3d,
        )
        diagnostics_2d = calculate_kinematics(
            ds_2d,
            time_dim_2d,
            z_dim_2d,
            selection_2d["time_index"],
            LEVEL_INDICES_2D,
            dx_2d,
            dy_2d,
        )

        active_fraction_3d = full_domain_active_fraction(diagnostics_3d["ro"])
        active_fraction_2d = full_domain_active_fraction(diagnostics_2d["ro"])

        vorticity_w1, vorticity_w1_normalized = distribution_distance(
            diagnostics_2d["ro"],
            diagnostics_3d["ro"],
        )
        strain_w1, strain_w1_normalized = distribution_distance(
            diagnostics_2d["strain_f"],
            diagnostics_3d["strain_f"],
        )

        divergence_proxy = divergence_rms_3d(diagnostics_3d["div_f"])

        selected_records.append(
            {
                "case_3d": CASE_3D,
                "case_2d": CASE_2D,
                "requested_day": requested_day,
                "actual_day_3d": selection_3d["actual_day"],
                "actual_day_2d": selection_2d["actual_day"],
                "time_index_3d": selection_3d["time_index"],
                "time_index_2d": selection_2d["time_index"],
                "active_fraction_3d": active_fraction_3d,
                "active_fraction_2d": active_fraction_2d,
                "active_fraction_error_signed": active_fraction_2d - active_fraction_3d,
                "active_fraction_error_absolute": abs(active_fraction_2d - active_fraction_3d),
                "vorticity_wasserstein": vorticity_w1,
                "vorticity_wasserstein_iqr_normalized": vorticity_w1_normalized,
                "strain_wasserstein": strain_w1,
                "strain_wasserstein_iqr_normalized": strain_w1_normalized,
                "divergence_rms_3d": divergence_proxy,
            }
        )

        selected_snapshot_data[requested_day] = {
            "ro_3d": diagnostics_3d["ro"],
            "ro_2d": diagnostics_2d["ro"],
            "strain_3d": diagnostics_3d["strain_f"],
            "strain_2d": diagnostics_2d["strain_f"],
            "div_3d": diagnostics_3d["div_f"],
            "dx_plot": min(dx_3d, dx_2d),
            "dy_plot": min(dy_3d, dy_2d),
        }

selected_metrics_df = pd.DataFrame(selected_records)
display(selected_metrics_df)

if SAVE_OUTPUTS:
    selected_metrics_df.to_csv(
        METRICS_DIR / "selected_case_snapshot_metrics.csv",
        index=False,
    )


In [ ]:
# ============================================================
# 6. METRIC OVERVIEW FIGURES
# ============================================================

fig, ax = plt.subplots(figsize=shu.get_figsize("wide"))

ax.plot(
    selected_metrics_df["requested_day"],
    selected_metrics_df["active_fraction_3d"],
    marker="o",
    label="3D",
)
ax.plot(
    selected_metrics_df["requested_day"],
    selected_metrics_df["active_fraction_2d"],
    marker="o",
    label="2D",
)

ax.set_xlabel("Time after production start [days]")
ax.set_ylabel(rf"Active area fraction, $|\zeta/f|>{ACTIVE_RO_THRESHOLD:g}$")
ax.set_ylim(bottom=0.0)
ax.set_title(f"Submesoscale-active area fraction | {CASE_3D} vs {CASE_2D}")
ax.legend()
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIG_DIR / "selected_case_active_fraction_snapshots.png",
        dpi=getattr(ptheme, "SAVE_DPI", 300),
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )

plt.show()


fig, ax = plt.subplots(figsize=shu.get_figsize("wide"))

ax.plot(
    selected_metrics_df["requested_day"],
    selected_metrics_df["vorticity_wasserstein_iqr_normalized"],
    marker="o",
    label=r"Vorticity, $\zeta/f$",
)
ax.plot(
    selected_metrics_df["requested_day"],
    selected_metrics_df["strain_wasserstein_iqr_normalized"],
    marker="o",
    label=r"Strain, $s/f$",
)

ax.set_xlabel("Time after production start [days]")
ax.set_ylabel(r"IQR-normalized Wasserstein distance, $W_1^*$")
ax.set_ylim(bottom=0.0)
ax.set_title(f"2D--3D distributional disagreement | {CASE_3D} vs {CASE_2D}")
ax.legend()
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIG_DIR / "selected_case_distribution_distances_snapshots.png",
        dpi=getattr(ptheme, "SAVE_DPI", 300),
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )

plt.show()


In [ ]:
# ============================================================
# 7. SELECTED-SNAPSHOT COMBINED PANELS
# ============================================================

for _, row in selected_metrics_df.iterrows():
    requested_day = float(row["requested_day"])
    snapshot = selected_snapshot_data[requested_day]
    day_tag = safe_filename_day(requested_day)

    ro_3d = snapshot["ro_3d"]
    ro_2d = snapshot["ro_2d"]
    strain_3d = snapshot["strain_3d"]
    strain_2d = snapshot["strain_2d"]
    dx_plot = snapshot["dx_plot"]
    dy_plot = snapshot["dy_plot"]

    if MAKE_SELECTED_CASE_MAPS:
        plot_side_by_side_field_panel(
            ro_3d,
            ro_2d,
            dx=dx_plot,
            dy=dy_plot,
            title_left="3D",
            title_right="2D",
            suptitle=(
                f"Normalized vorticity comparison | "
                f"{CASE_3D} vs {CASE_2D} | T={requested_day:g} d"
            ),
            cbar_label=r"$\zeta/f$",
            cmap="rossby",
            vmin=-2.0,
            vmax=2.0,
            output_path=(
                MAP_DIR / f"ro_panel_T{day_tag}d.png"
                if SAVE_FIGURES else None
            ),
        )

        vmin_strain = 0.0
        vmax_strain = float(
            np.nanpercentile(
                np.abs(
                    np.concatenate(
                        [
                            np.ravel(strain_3d[np.isfinite(strain_3d)]),
                            np.ravel(strain_2d[np.isfinite(strain_2d)]),
                        ]
                    )
                ),
                99.5,
            )
        )
        if not np.isfinite(vmax_strain) or vmax_strain <= 0.0:
            vmax_strain = 1.0

        plot_side_by_side_field_panel(
            strain_3d,
            strain_2d,
            dx=dx_plot,
            dy=dy_plot,
            title_left="3D",
            title_right="2D",
            suptitle=(
                f"Normalized strain comparison | "
                f"{CASE_3D} vs {CASE_2D} | T={requested_day:g} d"
            ),
            cbar_label=r"$s/f$",
            cmap="strain",
            vmin=vmin_strain,
            vmax=vmax_strain,
            output_path=(
                MAP_DIR / f"strain_panel_T{day_tag}d.png"
                if SAVE_FIGURES else None
            ),
        )

    if MAKE_SELECTED_CASE_PDFS:
        plot_combined_pdf_panel(
            ro_3d=ro_3d,
            ro_2d=ro_2d,
            strain_3d=strain_3d,
            strain_2d=strain_2d,
            metrics_row=row,
            title=(
                f"PDF comparison | {CASE_3D} vs {CASE_2D} | "
                f"T={requested_day:g} d"
            ),
            output_path=(
                PDF_DIR / f"pdf_panel_T{day_tag}d.png"
                if SAVE_FIGURES else None
            ),
        )

    if MAKE_SELECTED_CASE_JPDFS:
        plot_combined_jpdf_panel(
            ro_3d=ro_3d,
            strain_3d=strain_3d,
            ro_2d=ro_2d,
            strain_2d=strain_2d,
            title=(
                f"Vorticity--strain JPDF comparison | "
                f"{CASE_3D} vs {CASE_2D} | T={requested_day:g} d"
            ),
            output_path=(
                JPDF_DIR / f"jpdf_panel_T{day_tag}d.png"
                if SAVE_FIGURES else None
            ),
        )


In [ ]:
# ============================================================
# 8. CROSS-CASE SNAPSHOT METRICS
# ============================================================

if SUMMARY_CASES_3D is None:
    summary_cases_3d = discover_2d_3d_case_pairs(INPUT_DIR)
else:
    summary_cases_3d = list(SUMMARY_CASES_3D)

summary_cases_3d = [
    case_name
    for case_name in summary_cases_3d
    if case_name not in set(EXCLUDE_SUMMARY_CASES_3D)
]

all_case_records = []

for case_3d in summary_cases_3d:
    case_2d = f"{case_3d}_2D"

    path_3d = INPUT_DIR / f"{case_3d}.nc"
    path_2d = INPUT_DIR / f"{case_2d}.nc"

    case_config_3d = optional_case_config(case_3d)
    case_config_2d = optional_case_config(case_2d)

    ref_idx_3d = resolve_reference_index(case_config_3d, None)
    ref_idx_2d = resolve_reference_index(case_config_2d, None)

    with xr.open_dataset(path_3d, decode_times=False, cache=False) as ds_case_3d, xr.open_dataset(
        path_2d, decode_times=False, cache=False
    ) as ds_case_2d:

        time_dim_3d, z_dim_3d = dataset_layout(ds_case_3d)
        time_dim_2d, z_dim_2d = dataset_layout(ds_case_2d)

        dx_case_3d = infer_grid_spacing(ds_case_3d, "X", fallback=DX_3D)
        dy_case_3d = infer_grid_spacing(ds_case_3d, "Y", fallback=DY_3D)
        dx_case_2d = infer_grid_spacing(ds_case_2d, "X", fallback=DX_2D)
        dy_case_2d = infer_grid_spacing(ds_case_2d, "Y", fallback=DY_2D)

        time_days_3d = elapsed_days(ds_case_3d.sizes[time_dim_3d], DAY_PER_INDEX_3D, ref_idx_3d)
        time_days_2d = elapsed_days(ds_case_2d.sizes[time_dim_2d], DAY_PER_INDEX_2D, ref_idx_2d)

        selections_3d = nearest_snapshot_indices(time_days_3d, SNAPSHOT_DAYS, tolerance_days=SNAPSHOT_TOLERANCE_DAYS)
        selections_2d = nearest_snapshot_indices(time_days_2d, SNAPSHOT_DAYS, tolerance_days=SNAPSHOT_TOLERANCE_DAYS)

        for selection_3d, selection_2d in zip(selections_3d, selections_2d):
            requested_day = selection_3d["requested_day"]

            diagnostics_3d = calculate_kinematics(
                ds_case_3d,
                time_dim_3d,
                z_dim_3d,
                selection_3d["time_index"],
                LEVEL_INDICES_3D,
                dx_case_3d,
                dy_case_3d,
            )
            diagnostics_2d = calculate_kinematics(
                ds_case_2d,
                time_dim_2d,
                z_dim_2d,
                selection_2d["time_index"],
                LEVEL_INDICES_2D,
                dx_case_2d,
                dy_case_2d,
            )

            active_fraction_3d = full_domain_active_fraction(diagnostics_3d["ro"])
            active_fraction_2d = full_domain_active_fraction(diagnostics_2d["ro"])

            vorticity_w1, vorticity_w1_normalized = distribution_distance(
                diagnostics_2d["ro"],
                diagnostics_3d["ro"],
            )
            strain_w1, strain_w1_normalized = distribution_distance(
                diagnostics_2d["strain_f"],
                diagnostics_3d["strain_f"],
            )

            all_case_records.append(
                {
                    "case_3d": case_3d,
                    "case_2d": case_2d,
                    "requested_day": requested_day,
                    "active_fraction_3d": active_fraction_3d,
                    "active_fraction_2d": active_fraction_2d,
                    "active_fraction_error_signed": active_fraction_2d - active_fraction_3d,
                    "active_fraction_error_absolute": abs(active_fraction_2d - active_fraction_3d),
                    "vorticity_wasserstein": vorticity_w1,
                    "vorticity_wasserstein_iqr_normalized": vorticity_w1_normalized,
                    "strain_wasserstein": strain_w1,
                    "strain_wasserstein_iqr_normalized": strain_w1_normalized,
                    "divergence_rms_3d": divergence_rms_3d(diagnostics_3d["div_f"]),
                }
            )

all_case_snapshot_df = pd.DataFrame(all_case_records)
display(all_case_snapshot_df.head())

if SAVE_OUTPUTS:
    all_case_snapshot_df.to_csv(
        METRICS_DIR / "all_case_snapshot_metrics.csv",
        index=False,
    )


In [ ]:
# ============================================================
# 9. CROSS-CASE SUMMARY FIGURES
# ============================================================

fig, ax = plt.subplots(figsize=shu.get_figsize("square"))

minimum = float(
    np.nanmin(
        all_case_snapshot_df[["active_fraction_3d", "active_fraction_2d"]].to_numpy()
    )
)
maximum = float(
    np.nanmax(
        all_case_snapshot_df[["active_fraction_3d", "active_fraction_2d"]].to_numpy()
    )
)

padding = 0.05 * (maximum - minimum)
if not np.isfinite(padding) or padding == 0.0:
    padding = 0.01

line_min = max(0.0, minimum - padding)
line_max = maximum + padding

for requested_day in SNAPSHOT_DAYS:
    subset = all_case_snapshot_df[
        all_case_snapshot_df["requested_day"] == float(requested_day)
    ]
    ax.scatter(
        subset["active_fraction_3d"],
        subset["active_fraction_2d"],
        label=f"T={requested_day:g} d",
    )

ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", label="1:1")

ax.set_xlim(line_min, line_max)
ax.set_ylim(line_min, line_max)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(r"$A_{\mathrm{sub}}^{3D}$")
ax.set_ylabel(r"$A_{\mathrm{sub}}^{2D}$")
ax.set_title("Submesoscale-active area fraction at selected snapshots")
ax.legend()
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIG_DIR / "all_cases_active_fraction_paired_snapshots.png",
        dpi=getattr(ptheme, "SAVE_DPI", 300),
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )

plt.show()


In [ ]:
# ============================================================
# 10. OUTPUT CHECK
# ============================================================

required_selected_columns = [
    "requested_day",
    "active_fraction_3d",
    "active_fraction_2d",
    "vorticity_wasserstein_iqr_normalized",
    "strain_wasserstein_iqr_normalized",
    "divergence_rms_3d",
]

missing_selected = [
    column
    for column in required_selected_columns
    if column not in selected_metrics_df.columns
]

if missing_selected:
    raise RuntimeError(
        f"Missing selected-case output columns: {missing_selected}"
    )

print("Selected-case snapshots:")
display(selected_metrics_df[required_selected_columns])

print(f"Saved metrics to: {METRICS_DIR}")
print(f"Saved figures to: {FIG_DIR}")
print("Done.")
